# NCM-GPT1 在 Kaggle 訓練（T4 × 2）

設定（右上角）：
- Accelerator: **GPU T4 x2**
- Internet: **On**
- 用 **Save & Run All**（不要用互動模式長跑，閒置約 20 分鐘會被中斷）

每一場最多 12 小時。不要重裝 PyTorch。

**不要中斷正在跑的 kernel v8。** v8 結束後，下一場請掛上 **COMPLETE 的 kernel 第 8 版 Output**（tokenizer 已完成；generator 從 v8 的 `latest.pt` 續訓，v8 是從 step 40457 接著跑的）。`finished: true` 的 tokenizer 會被跳過。

T4（sm_75）維持梯度檢查點、chunked block attention、打包時輪詢 `packed_windows.pt`（不要用 NCCL barrier），以及 Ampere 以前的 fp16 + GradScaler（不要用模擬 bf16）。`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`。

這一版給 v8 **之後** 的下一場，不是給正在跑的 v8：
- 每卡 batch **2**。v7 在 batch 4 時 tokens/s 仍約 1400（和 v5 batch 2 一樣），steps/s 從約 0.69 降到約 0.34，峰值顯存約 4.1 GiB、沒有 OOM。步數預算用 batch 2 更快。梯度檢查點維持開啟。
- `alpha: 0.5` 維持（已偏離論文 ELBO 的 alpha 0）。另外 `mid_aux: 0.5` 把預測層中央三分之一（長度約 81–225，v7 的 CE 約 5.2–6.1）的平均 CE 加進總損失。總損失不要和 v5、v7 或 v8 比較；`ce_first` / `ce_mid` / `ce_last` / `ce_levels` 仍可比較。
- `max_windows: 524288` 與 `packed_windows.pt` 快取維持。v8 已 cache hit，下一場應繼續重用。

若這份程式還沒在 `main` 上，請把含這些改動的倉庫當成 Input dataset 掛上，不要讓 clone 到的舊 `main` 蓋掉它。

In [ ]:
# 只補訓練依賴，不要重裝 torch
import os, subprocess, sys
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "omegaconf", "hydra-core", "datasets", "regex", "tqdm"])
print("deps ok")

In [ ]:
import os, sys, subprocess
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

REPO = "https://github.com/nicholashui/ncm-gpt1.git"
ROOT = Path("/kaggle/working/ncm-gpt1")
if not (ROOT / "train_tokenizer.py").exists():
    # 若已把程式當成 Input dataset 掛進來，直接複製；否則 clone。
    # 在這份改動進入 main 之前，請掛含 batch 2 / mid_aux 0.5 / max_windows 524288 的程式，避免 clone 到舊 main。
    local = None
    for cand in Path("/kaggle/input").glob("*/train_tokenizer.py") if Path("/kaggle/input").exists() else []:
        local = cand.parent
        break
    if local:
        import shutil
        shutil.copytree(local, ROOT, dirs_exist_ok=True)
        print("copied repo from", local)
    else:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO, str(ROOT)])
        print("cloned", REPO)
else:
    print("repo already present")

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("cwd", Path.cwd())

In [ ]:
import torch
print("cuda:", torch.cuda.is_available(), "gpus:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        cap = torch.cuda.get_device_capability(i)
        print(i, torch.cuda.get_device_name(i), "mem", round(torch.cuda.get_device_properties(i).total_memory/1024**3, 1), "GiB", "cap", cap)
assert torch.cuda.is_available(), "請把 Accelerator 設成 GPU T4 x2"
assert torch.cuda.device_count() >= 1

訓練語料預設是開源 BookCorpus 重建版（與 2018 GPT-1 的書籍語料同一類）。若下載失敗，腳本會改用 MiniPile，再不行才用 TinyStories。檢查點寫到 `/kaggle/working/artifacts`。

v8 結束後掛上該場完整 Output。tokenizer 會被跳過，generator 從 v8 的 `latest.pt` 載入權重與 optimizer。下一場使用 batch 2、`alpha: 0.5`、`mid_aux: 0.5`。規則切分器沒有權重，仍會按句打包，並寫下 `artifacts/splitter/status.json`。

In [ ]:
import os, subprocess, sys
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

out = Path("/kaggle/working/artifacts")
out.mkdir(parents=True, exist_ok=True)

# 若上一場的 Output 已掛成 Input，自動接續。
# v8 結束後掛 COMPLETE kernel v8 Output。不要中斷正在跑的 v8。
cmd = [
    sys.executable, "scripts/train_kaggle.py",
    "--hours", "11.5",
    "--reserve_minutes", "15",
    "--output_dir", str(out),
    "--config", "config/kaggle.yaml",
    "--batch_size", "2",
    "--gradient_checkpointing",
]
print("running:", " ".join(cmd), flush=True)
rc = subprocess.call(cmd)
print("exit", rc)
if rc != 0:
    raise SystemExit(rc)

In [ ]:
from pathlib import Path

def show(stage):
    p = Path("/kaggle/working/artifacts") / stage / "status.json"
    print("=" , stage, "=")
    if p.exists():
        print(p.read_text())
    else:
        print("no status.json yet")
    log = Path("/kaggle/working/artifacts") / stage / f"{stage}_log.jsonl"
    if log.exists():
        lines = log.read_text(encoding="utf-8").strip().splitlines()
        print("log lines", len(lines))
        if lines:
            print("last:", lines[-1])

show("tokenizer")
show("generator")